# Atlas Trade — Unsloth Qwen3.5-27B Setup

**Runtime:** Google Colab Pro — A100 80GB  
**Model:** `unsloth/Qwen3.5-27B` (BF16, no quantization)  
**Context:** 16384 tokens

Run cells top to bottom on a fresh A100 runtime. On subsequent sessions, re-run from **Step 4** onwards (model is cached in HF cache after the first download).

## Configuration

Set your GitHub repo URL and Hugging Face token here, or export them as environment variables before running:

```python
import os
os.environ["PROJECT_REPO_URL"] = "https://github.com/<your-username>/unsloth-qwen35-trading-assistant.git"
os.environ["HF_TOKEN"] = "hf_..."
```

In [1]:
import os

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# ── Model seçimi ────────────────────────────────────────────────────────────
# Qwen3-32B: Qwen3.5-27B'ye göre daha güçlü reasoning, native thinking modu,
#            A100 80GB'de 4-bit ile ~18GB VRAM kullanır.
MODEL_NAME     = "unsloth/Qwen3-32B"
MAX_SEQ_LENGTH = 16384   # 16k context — reasoning zinciri için gerekli
DTYPE          = None    # auto-detect (BF16 on A100)
LOAD_IN_4BIT   = True    # ~18GB VRAM

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"

print(f"Model   : {MODEL_NAME}")
print(f"Context : {MAX_SEQ_LENGTH} tokens")
print(f"4-bit   : {LOAD_IN_4BIT}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {'set' if HF_TOKEN else 'not set'}")

Model   : unsloth/Qwen3-32B
Context : 16384 tokens
4-bit   : True
Repo    : https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git
HF token: not set


## Step 1 — Clone or update the repository

In [2]:
import subprocess, pathlib, sys

def _run(cmd: list[str], **kwargs) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    # Add repo to Python path so `src` is importable
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

Cloning into '/content/unsloth-qwen35-trading-assistant'...
Repo ready at /content/unsloth-qwen35-trading-assistant


## Step 2 — Install Unsloth

Uses the official runtime-aware auto-installer — automatically matches the active CUDA/Torch stack on this Colab runtime.

In [3]:
import subprocess, sys

def _pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args],
        capture_output=True, text=True
    )
    # Print last 2000 chars of output so failures are visible
    out = (result.stdout + result.stderr)[-2000:]
    print(out)
    return result.returncode

print("=== Installing Unsloth ===\n")
code = _pip("install", "unsloth")

if code != 0:
    print("\nStandard install failed, trying git source...")
    code = _pip(
        "install",
        "unsloth @ git+https://github.com/unslothai/unsloth.git",
        "--no-build-isolation",
    )

if code != 0:
    raise RuntimeError("Unsloth installation failed. See output above.")

print("\n✅ Unsloth installed successfully.")
print("\n⚠️  RESTART THE RUNTIME NOW, then re-run from Step 1 (skip this cell).")
print("   Runtime menu → Restart session")

=== Installing Unsloth ===

━━━━━━━ 3.3/3.3 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 29.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing

## Step 3 — Hugging Face login

Required only if you plan to push LoRA adapters or access gated models. Skip if `HF_TOKEN` is empty.

In [4]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set — skipping login (public model download will still work).")

No HF_TOKEN set — skipping login (public model download will still work).


## Step 4 — Load the model

First run downloads ~54 GB of BF16 weights. Subsequent runs load from the HF cache (~1-2 minutes).

In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-32b-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded: unsloth/Qwen3-32B
Max sequence length: 16384


## Step 5 — Chat function

`chat()` keeps a conversation history so Atlas Trade remembers the context within a session.  
Call `chat(reset=True)` to start a fresh conversation.

In [6]:
import sys, re
sys.path.insert(0, REPO_DIR)

from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages

_conversation_history = []
_text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer


def _strip_thinking(text: str) -> str:
    """
    Qwen3 <think>...</think> bloğunu temizle, sadece asıl cevabı döndür.
    Thinking içeriği tamamen kaldırılır — kullanıcıya sade, temiz çıktı gider.
    """
    # <think>...</think> bloğunu (multiline) kaldır
    cleaned = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.DOTALL)
    return cleaned.strip()


def chat(
    user_message: str,
    market_context: str = "",
    reset: bool = False,
    thinking: bool = True,         # ← varsayılan AÇIK: model önce düşünür, sonra yazar
    max_new_tokens: int = 2048,    # ← düşünce zinciri için artırıldı
    temperature: float = 0.6,
    top_p: float = 0.95,
):
    """
    Atlas Trade ile sohbet et.

    Parametreler
    ------------
    user_message    : Sorun veya analiz isteğin.
    market_context  : fetch_context() veya fetch_multi_tf_context() çıktısı.
    reset           : True → geçmiş sıfırlanır, yeni konuya başlanır.
    thinking        : True → model önce sessiz muhakeme yapar (önerilen).
    max_new_tokens  : Maksimum üretilecek token (thinking dahil).
    """
    global _conversation_history
    if reset:
        _conversation_history = []
        print("[sıfırlandı]")

    messages = build_trading_messages(
        user_message=user_message,
        market_context=market_context,
        history=_conversation_history,
    )

    # Chat template — enable_thinking Qwen3 serisinde desteklenir
    try:
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=thinking,
        )
    except TypeError:
        # Eski transformers sürümü enable_thinking bilmiyorsa sessizce geç
        text = _text_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    inputs = _text_tokenizer(text, return_tensors="pt").to("cuda")

    outputs = model.generate(
        input_ids      = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        pad_token_id   = _text_tokenizer.eos_token_id,
        max_new_tokens = max_new_tokens,
        temperature    = temperature,
        top_p          = top_p,
        do_sample      = True,
    )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw        = _text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response   = _strip_thinking(raw)

    _conversation_history.append({"role": "user",      "content": user_message})
    _conversation_history.append({"role": "assistant", "content": response})
    return response


def clear_history():
    global _conversation_history
    _conversation_history = []
    print("[geçmiş temizlendi]")


print("✅ chat() hazır")
print("   thinking modu: AÇIK (varsayılan)")
print("   max_new_tokens: 2048")

✅ chat() hazır
   thinking modu: AÇIK (varsayılan)
   max_new_tokens: 2048


## Step 6 — Start trading

Atlas Trade hazır. Aşağıdaki hücreyi düzenleyerek analizini başlat.

## Step 7 — Live market data (Bybit)

`fetch_context()` Bybit'ten canlı OHLCV çeker, RSI / EMA / MACD hesaplar ve `chat()` için hazır bir `market_context` stringi döner.  
API key gerekmez — public endpoint.

In [7]:
import importlib
import src.market_data as _md
importlib.reload(_md)
from src.market_data import fetch_context, fetch_multi_tf_context


def live_chat(
    user_message: str,
    symbol: str = "BTCUSDT",
    timeframe: str = "4h",
    multi_tf: bool = False,
    timeframes: list = None,
    reset: bool = False,
    **kwargs,
):
    """
    Bybit'ten canlı piyasa verisi çekip otomatik olarak chat()'e gönderir.

    Parametreler
    ------------
    user_message : Analiz sorun.
    symbol       : Bybit sembolü, örn. "BTCUSDT", "ETHUSDT", "SOLUSDT"
    timeframe    : Tek TF modu için: "1m","5m","15m","30m","1h","4h","1d"
    multi_tf     : True → birden fazla TF çekilir (daha kapsamlı analiz).
    timeframes   : multi_tf=True iken kullanılacak TF listesi.
                   Varsayılan: ["1d", "4h", "1h"]
    reset        : True → konuşma geçmişini sıfırla.

    Örnekler
    --------
    live_chat("BTC long setup var mı?")
    live_chat("ETH analiz et", symbol="ETHUSDT", timeframe="1h")
    live_chat("SOL yapısı nasıl?", symbol="SOLUSDT", multi_tf=True)
    live_chat("BTC tam analiz", multi_tf=True, timeframes=["1d","4h","1h","15m"])
    """
    if multi_tf:
        tfs = timeframes or ["1d", "4h", "1h"]
        print(f"[{symbol}  {' + '.join(t.upper() for t in tfs)}  verisi alınıyor...]")
        ctx = fetch_multi_tf_context(symbol, timeframes=tfs)
    else:
        print(f"[{symbol} {timeframe.upper()} verisi alınıyor...]")
        ctx = fetch_context(symbol, timeframe=timeframe)

    print(ctx)
    print("─" * 60)
    return chat(user_message, market_context=ctx, reset=reset, **kwargs)


print("✅ live_chat() hazır")
print()
print("  Tek TF  : live_chat('BTC long uygun mu?')")
print("  Çoklu TF: live_chat('BTC yapısı?', multi_tf=True)")
print("  ETH     : live_chat('ETH analiz', symbol='ETHUSDT', timeframe='1h')")
print("  Full    : live_chat('Tam analiz', multi_tf=True, timeframes=['1d','4h','1h','15m'])")

✅ live_chat() hazır

  Tek TF  : live_chat('BTC long uygun mu?')
  Çoklu TF: live_chat('BTC yapısı?', multi_tf=True)
  ETH     : live_chat('ETH analiz', symbol='ETHUSDT', timeframe='1h')
  Full    : live_chat('Tam analiz', multi_tf=True, timeframes=['1d','4h','1h','15m'])


In [8]:
# --- Sohbet kutusu ---
# Mesajini asagiya yaz, hucreyi calistir. Tekrar yaz, tekrar calistir.
# Yeni konuya gecmek icin reset=True yap.

mesaj = "BTC 73k'yı kırarsa ne olur sence?"

response = chat(mesaj, reset=True)
print(response)

[sıfırlandı]


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

**📊 Trend Rejimi**  
Haftalık ve günlük vadede BTC, $63k-$73k aralığında 3 aylık bir trend kanalında hareket ediyor. 4H grafikte $73k direnci kırılırsa trend kanalı pozitif yönde genişleyebilir. Ancak 1H'de fiyat, $73k seviyesinde sıkışmış durumda ve breakout onayı bekleniyor.  

**🏗️ Yapı & Seviyeler**  
- Kritik direnç: **$73,000** (son 3 ayın yüksekliği)  
- Kritik destek: **$68,000** (önceki seviye, likidasyon havuzu)  
- Tetik seviyesi: **$73,500** (kapanış kırılması)  
- İptal seviyesi: **$72,000** (breakout kırılırsa senaryo geçersiz olur)  

**⚡ Momentum & Piyasa Yapısı**  
- RSI: 68 seviyesinde (aşırı alım değil, ancak kırılma sonrası momentum ivmelenebilir).  
- MACD: Histogram pozitif yönde genişleyerek yukarı yönlü ivme veriyor.  
- Hacim: 4H'de son 24 saatte %20 artış var, ancak $73k seviyesinde yoğun satış gölgesi var.  
- Funding: Şu anki pozisyonlar %0.048 (bullish), ancak aşırı pozitif değil.  

**🎯 Senaryo A — Bullish**  
- Koşul: $73,500'de kapanış kırılması ve 4H'de

---

## Utility — Update repo without reloading the model

In [11]:
# Run this cell whenever you push prompt or code changes from VS Code.
# The model stays loaded — only the Python source is refreshed.
git_pull_latest()

# Reload the trading_prompt module to pick up any changes
import importlib, src.trading_prompt as _tp
importlib.reload(_tp)
from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages
print("Prompt module reloaded.")

Updating 2c2bda6..bfbeea8
Fast-forward
 setup.ipynb | 665 +++++++++++++++++++++++++++++++++++++++++++++++++++++++-----
 1 file changed, 611 insertions(+), 54 deletions(-)
From https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant
   2c2bda6..bfbeea8  main       -> origin/main
Repo ready at /content/unsloth-qwen35-trading-assistant
Prompt module reloaded.


---
## Step 8 — Atlas Trade Panel (Gradio UI)

Hücreyi çalıştır → birkaç saniye sonra `https://xxxx.gradio.live` çıkar → tıkla → panel açılır.

In [12]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "gradio", "-q"], capture_output=True)

import gradio as gr
import importlib
import src.market_data as _md
importlib.reload(_md)
from src.market_data import fetch_context, fetch_multi_tf_context

# ── Sabit listeler ────────────────────────────────────────────────────────
POPÜLER_COİNLER = [
    "BTCUSDT", "ETHUSDT", "SOLUSDT", "BNBUSDT", "XRPUSDT",
    "DOGEUSDT", "AVAXUSDT", "ADAUSDT", "DOTUSDT", "LINKUSDT",
    "MATICUSDT", "LTCUSDT", "ATOMUSDT", "NEARUSDT", "APTUSDT",
]

TIMEFRAME_SEÇENEKLERİ = ["15m", "30m", "1h", "4h", "1d"]

HAZIR_SORULAR = [
    "Bu sembol için long setup var mı?",
    "Kısa vadeli trend ne yönde?",
    "Güçlü bir destek/direnç seviyesi var mı?",
    "Entry, stop ve hedef ver.",
    "Mevcut momentum devam eder mi?",
    "Short için uygun bir setup var mı?",
    "Risk/reward açısından şu an işlem değer mi?",
]

# ── Ana analiz fonksiyonu ─────────────────────────────────────────────────
def analiz_et(sembol, özel_sembol, timeframe, multi_tf_aktif, soru, geçmişi_sıfırla):
    """Gradio arayüzünden gelen isteği işler ve analiz döndürür."""

    # Özel sembol yazıldıysa onu kullan, yoksa dropdown seçimini al
    hedef_sembol = özel_sembol.strip().upper() if özel_sembol.strip() else sembol
    if not hedef_sembol.endswith("USDT"):
        hedef_sembol += "USDT"

    if not soru.strip():
        soru = "Bu sembol için kapsamlı bir teknik analiz yap."

    durum = f"⏳ {hedef_sembol} verisi alınıyor..."
    yield durum, ""

    try:
        if multi_tf_aktif:
            ctx = fetch_multi_tf_context(hedef_sembol, timeframes=["1d", "4h", "1h"])
            veri_başlık = f"📡 {hedef_sembol} — 1D + 4H + 1H verisi alındı"
        else:
            ctx = fetch_context(hedef_sembol, timeframe=timeframe)
            veri_başlık = f"📡 {hedef_sembol} {timeframe.upper()} verisi alındı"
    except Exception as e:
        yield f"❌ Veri alınamadı: {e}", ""
        return

    durum = f"{veri_başlık}\n🤔 Model analiz yapıyor..."
    yield durum, ""

    try:
        yanıt = chat(
            user_message=soru,
            market_context=ctx,
            reset=geçmişi_sıfırla,
        )
    except Exception as e:
        yield f"❌ Model hatası: {e}", ""
        return

    yield f"✅ {veri_başlık} — Analiz tamamlandı", yanıt


# ── Gradio arayüzü ────────────────────────────────────────────────────────
with gr.Blocks(
    title="Atlas Trade",
    theme=gr.themes.Base(
        primary_hue="emerald",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Inter"),
    ),
    css="""
        .output-box textarea { font-size: 15px !important; line-height: 1.7 !important; }
        .status-box textarea { font-size: 12px !important; color: #94a3b8 !important; }
        footer { display: none !important; }
        #baslik { text-align: center; padding: 16px 0 8px 0; }
    """,
) as panel:

    gr.Markdown("# Atlas Trade\n**Kripto Vadeli İşlemler — AI Teknik Analiz Paneli**", elem_id="baslik")

    with gr.Row():
        # ── Sol sütun: Girdiler ──────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown("### Sembol")
            sembol_dd = gr.Dropdown(
                choices=POPÜLER_COİNLER,
                value="BTCUSDT",
                label="Popüler coinler",
                interactive=True,
            )
            özel_sembol = gr.Textbox(
                placeholder="Farklı bir coin: örn. SUIUSDT",
                label="Veya kendin yaz (USDT otomatik eklenir)",
                max_lines=1,
            )

            gr.Markdown("### Zaman Dilimi")
            timeframe_dd = gr.Dropdown(
                choices=TIMEFRAME_SEÇENEKLERİ,
                value="4h",
                label="Timeframe",
                interactive=True,
            )
            multi_tf = gr.Checkbox(
                label="Çoklu TF analizi (1D + 4H + 1H)",
                value=False,
                info="Daha kapsamlı ama biraz daha yavaş",
            )

            gr.Markdown("### Soru")
            hazır_soru_dd = gr.Dropdown(
                choices=HAZIR_SORULAR,
                value=None,
                label="Hazır sorular",
                interactive=True,
            )
            soru_kutusu = gr.Textbox(
                placeholder="Veya kendi sorunuzu yazın...",
                label="Serbest soru",
                lines=3,
            )

            sıfırla_cb = gr.Checkbox(
                label="Konuşma geçmişini sıfırla",
                value=True,
                info="Her yeni analizde işaretli bırakmanız önerilir",
            )

            analiz_btn = gr.Button("ANALİZ ET", variant="primary", size="lg")

        # ── Sağ sütun: Çıktılar ──────────────────────────────────
        with gr.Column(scale=2):
            durum_kutusu = gr.Textbox(
                label="Durum",
                lines=2,
                interactive=False,
                elem_classes=["status-box"],
            )
            sonuç_kutusu = gr.Textbox(
                label="Atlas Trade Analizi",
                lines=28,
                interactive=False,
                elem_classes=["output-box"],
                show_copy_button=True,
            )

    # Hazır soru seçilirse soru kutusunu doldur
    hazır_soru_dd.change(
        fn=lambda s: s or "",
        inputs=hazır_soru_dd,
        outputs=soru_kutusu,
    )

    # Analiz butonu
    analiz_btn.click(
        fn=analiz_et,
        inputs=[sembol_dd, özel_sembol, timeframe_dd, multi_tf, soru_kutusu, sıfırla_cb],
        outputs=[durum_kutusu, sonuç_kutusu],
    )

    gr.Markdown(
        "_Atlas Trade bir karar destek aracıdır. Finansal tavsiye değildir. "
        "Tüm işlemler kişisel sorumluluğunuzdadır._",
    )

print("🚀 Panel başlatılıyor...")
panel.queue().launch(share=True, quiet=True)
print("✅ Panel hazır — yukarıdaki gradio.live linkine tıkla")

/tmp/ipykernel_3542/609020704.py:72: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3542/609020704.py:72: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


🚀 Panel başlatılıyor...
* Running on public URL: https://d0164ae4c2a7fb2fd0.gradio.live


✅ Panel hazır — yukarıdaki gradio.live linkine tıkla


Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[sıfırlandı]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[sıfırlandı]


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2048) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.1

---
## Step 9 — Google Drive Bağlantısı (Duman Köprüsü)

Google Drive'ı mount et, `oscar_duman` klasörünü bağla. **Bir kez** çalıştır.

In [ ]:
from google.colab import drive
import json, os
from pathlib import Path
from datetime import datetime, timezone

drive.mount("/content/drive", force_remount=False)

OSCAR_DUMAN_DIR = Path("/content/drive/MyDrive/oscar_duman")
OSCAR_DUMAN_DIR.mkdir(exist_ok=True)

def _load(fname):
    p = OSCAR_DUMAN_DIR / fname
    if not p.exists():
        return [] if fname == "trade_history.json" or fname == "reflections.json" else {}
    with open(p, encoding="utf-8") as f:
        return json.load(f)

def _save(fname, data):
    p = OSCAR_DUMAN_DIR / fname
    tmp = p.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    tmp.replace(p)

def load_market_data():   return _load("market_data.json")
def load_active_signals(): return _load("active_signals.json")
def load_trade_history():
    d = _load("trade_history.json")
    return d if isinstance(d, list) else []
def save_signal(signal):
    signals = load_active_signals()
    signals[signal["sinyal_id"]] = signal
    _save("active_signals.json", signals)
def save_reflection(ref):
    refs = load_trade_history()
    refs2 = _load("reflections.json")
    if not isinstance(refs2, list): refs2 = []
    refs2.append(ref)
    _save("reflections.json", refs2)

print(f"Drive baglandi: {OSCAR_DUMAN_DIR}")
md = load_market_data()
guncelleme = md.get("_guncelleme", "yok")
print(f"Piyasa verisi : {len(md)} coin  |  Guncelleme: {guncelleme}")
print(f"Aktif sinyal  : {len(load_active_signals())}")
print(f"Islem gecmisi : {len(load_trade_history())}")


---
## Step 10 — Oscar Sinyal Üreticisi

Oscar analiz yapar, sinyali parse eder, Drive'a yazar. Duman oradan alır.

```python
oscar_signal('BTCUSDT')                        # tek coin
oscar_signal('ETHUSDT', timeframe='1h')        # özel TF
```

In [ ]:
import re
from datetime import datetime, timezone

def _utcnow():
    return datetime.now(timezone.utc).isoformat()

def _to_float(s):
    s = str(s).strip().replace(",", "").replace("$", "").replace(" ", "")
    if s.lower().endswith("k"): return float(s[:-1]) * 1000
    if s.lower().endswith("m"): return float(s[:-1]) * 1_000_000
    return float(s)

def _parse_signal(analysis: str, symbol: str) -> dict | None:
    """Oscar analiz metninden entry/stop/TP1/TP2 cıkar."""
    NUM = r"[$]?[\d][\d,.]*[kKmM]?"

    # Yon
    yon = None
    text_lower = analysis.lower()
    long_score  = text_lower.count("long") + text_lower.count("bullish")
    short_score = text_lower.count("short") + text_lower.count("bearish")
    if long_score > short_score:   yon = "LONG"
    elif short_score > long_score: yon = "SHORT"
    if yon is None:
        return None

    def find(patterns):
        for pat in patterns:
            m = re.search(pat, analysis, re.IGNORECASE)
            if m:
                try: return _to_float(m.group(1))
                except: pass
        return None

    entry = find([
        r"[Ee]ntry[:\s*:]+(" + NUM + ")",
        r"[Gg]iri[sş][:\s*:]+(" + NUM + ")",
        r"[Ee]ntry.*?(" + NUM + ")",
    ])
    stop = find([
        r"[Ss]top[\s*:]+(" + NUM + ")",
        r"\bSL[\s*:]+(" + NUM + ")",
        r"[İI]ptal[\s*:]+(" + NUM + ")",
        r"[Ss]top.*?(" + NUM + ")",
    ])
    tp1 = find([
        r"[Hh]edef\s*1[\s*:]+(" + NUM + ")",
        r"TP\s*1[\s*:]+(" + NUM + ")",
        r"[Hh]edef[\s*:]+(" + NUM + ")",
    ])
    tp2 = find([
        r"[Hh]edef\s*2[\s*:]+(" + NUM + ")",
        r"TP\s*2[\s*:]+(" + NUM + ")",
    ])

    if not all([entry, stop, tp1]):
        return None

    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    return {
        "sinyal_id":          f"{symbol}_{ts}",
        "symbol":             symbol,
        "yon":                yon,
        "entry":              entry,
        "stop":               stop,
        "hedef_1":            tp1,
        "hedef_2":            tp2 or tp1,
        "durum":              "bekliyor",
        "analiz":             analysis,
        "olusturma_zamani":   _utcnow(),
    }


def _build_context_from_cache(symbol: str) -> str:
    """market_data.json icindeki veriyi Oscar icin formatla."""
    md = load_market_data()
    if symbol not in md:
        return ""
    d = md[symbol]
    lines = [f"Sembol: {symbol}"]
    for tf, tfd in d.get("timeframes", {}).items():
        lines.append(f"\n── {tf.upper()} ─────────────")
        lines.append(f"Fiyat   : {tfd.get('price', 0):.2f}")
        lines.append(f"RSI14   : {tfd.get('rsi14', 0):.1f}")
        lines.append(f"ATR14   : {tfd.get('atr14', 0):.4f}")
        lines.append(f"EMA bias: {tfd.get('ema_bias', '?')}")
        lines.append(f"MACD hist: {tfd.get('macd_hist', 0):.4f}")
    try:
        funding_pct = float(d.get("funding_rate", 0)) * 100
        lines.append(f"\nFunding : {funding_pct:.4f}%")
    except Exception:
        pass
    lines.append(f"OI      : {d.get('open_interest', 'N/A')}")
    lines.append(f"Veri ts : {d.get('ts', 'N/A')}")
    return "\n".join(lines)


def oscar_signal(
    symbol: str,
    timeframe: str = "4h",
    soru: str = "",
) -> dict | None:
    """
    Tek coin icin Oscar analiz uretir, sinyali Drive'a yazar.
    Once market_data.json'daki cache'i kullanir,
    yoksa canli Bybit verisi ceker.
    """
    if not soru:
        soru = (
            "Bu sembol icin net bir trade sinyali ver. "
            "Entry, stop ve iki hedef seviyesi belirt."
        )

    ctx = _build_context_from_cache(symbol)
    if not ctx:
        from src.market_data import fetch_multi_tf_context
        print(f"[{symbol}] Cache yok — canli veri cekiliyor...")
        ctx = fetch_multi_tf_context(symbol)

    print(f"[{symbol}] Oscar analiz yapiyor...")
    analiz = chat(soru, market_context=ctx, reset=True)
    print("\n" + analiz)
    print("─" * 55)

    sinyal = _parse_signal(analiz, symbol)
    if sinyal:
        save_signal(sinyal)
        print(f"\n✅ Sinyal Drive'a yazildi: {sinyal['sinyal_id']}")
        print(f"   {sinyal['yon']}  Entry:{sinyal['entry']}  "
              f"Stop:{sinyal['stop']}  TP1:{sinyal['hedef_1']}  TP2:{sinyal['hedef_2']}")
    else:
        print("\n⚠️  Sinyal parse edilemedi — analiz entry/stop/hedef icermiyor.")
        print("   Ipucu: Soruyu daha net sor veya soru='Entry, stop ve hedef ver.' kullan.")
    return sinyal


print("✅ oscar_signal() hazir")
print("   Kullanim:")
print("   oscar_signal('BTCUSDT')")
print("   oscar_signal('ETHUSDT', timeframe='1h')")
